# **PRACTICE 6: Neural Language Model (Part 1)**

There are aggressive and non-aggressive tweets in the training and validation files, as well as their labels. Here, an example of classification is given.

In [1]:
# Libraries.       
import nltk                                            # library for text processing.
from nltk.tokenize import TweetTokenizer               # library for tweet tokenization.
from nltk.corpus import stopwords                      # library for stopwords.
stopwords_es = set(stopwords.words('spanish'))         # set of spanish stopwords.
import numpy as np                                     # library for numerical operations.
import os                                              # library for operating system.
import time                                            # library for time.
import shutil                                          # library for file operations.
from typing import Tuple                               # library for typing.
import random                                          # library for random numbers.
import pandas as pd                                    # library for data manipulation.
import torch                                           # library for tensor operations.
from tqdm import tqdm                                  # library for progress bar.
import torch.nn as nn                                  # library for neural network operations.
from torch.utils.data import DataLoader, TensorDataset # library for data loading.
import torch.nn.functional as F                        # library for neural network operations.
from sklearn.metrics import accuracy_score             # library for accuracy score.
from itertools import permutations                     # library for permutations.
np.random.seed(0)                                      # seed for reproducibility.
random.seed(0)                                         # seed for reproducibility.
torch.manual_seed(0)                                   # seed for reproducibility.
torch.backends.cudnn.benchmark = False                 # reproducibility.

First we charge de training and validation data in list form (list with every tweet as a string). Every element is a tweet. 

In [2]:
X_train = pd.read_csv('mex20_train.txt', sep = '\r\n', header = None, engine = 'python').loc[:,0].values.tolist()
X_val   = pd.read_csv('mex20_val.txt', sep = '\r\n', header = None, engine = 'python').loc[:,0].values.tolist()
print("First 5 elements in Train data:\n", X_train[:5])
print("First 5 elements in Validation data:\n", X_val[:5])

First 5 elements in Train data:
 ['@USUARIO @USUARIO @USUARIO Q se puede esperar del maricon de closet de la Yañez aun recuerdo esa ves q lo vi en zona rosa viendo quien lo levantada', '@USUARIO La piel nueva siempre arde un poquito los primeros días... y más con este puto clima', 'Ustedes no se enamoran de mí… por tontas.', 'Me las va a pagar esa puta gorda roba tuits...', '@USUARIO LA GENTE ES TONTA PORQUE NO SE DAN CUENTA QUE TÚ HACES A BATMAN AZUL']
First 5 elements in Validation data:
 ['Al perro que se te acerque le parto su madre a si de facil', '@USUARIO @USUARIO Él supo sacar a su familia adelante en lo que sabe en el mundo existen muchas personas ardidas como tú 🤷🏻\u200d♀️', '@USUARIO Entonces para que quieres estar en sus paises?, mejor vente aca y chinguele cabrona de verdad, maldita sangana.', 'Que bueno que hoy juega México, porque tú vales verga, hija de la chingada.', 'Ojalá un día me valgas la misma verga que al vato de la cfe despelucando y ponchando a medio del puent

# **1.- NgramData**

In [3]:
class NgramData:
    """
    N-gram data preparation class. It prepares the data for the n-gram model.
    """
    def __init__(self, model_order: int, max_vocab_size: int = 5000,
                 tokenizer: TweetTokenizer = None, embeddings_model: np.ndarray = None,
                 remove_stopwords: bool = False):
        """
        Constructor method. If no tokenizer is provided, the default tokenizer is used. If no embeddings model is provided, random embeddings are used.
        
        Parameters
        ----------
        model_order : int
            Order of the n-gram model.
        max_vocab_size : int, optional
            Maximum vocabulary size (default is 5000).
        tokenizer : TweetTokenizer, optional
            Tokenizer object. Default is None.
        embeddings_model : np.ndarray, optional
            Embeddings model. Default is None. An embeddings model is a dictionary where the keys are words and the values are the embeddings.
        remove_stopwords : bool, optional
            Remove stopwords. Default is False.
        """
        self.tokenizer = tokenizer or self.default_tokenizer  # tokenizer object.
        self.punct = {'.',',',';',':','!','?','¿','¡','(',')','[',']','{','}',          # punctuation.
                      '"',"'",'...','…','“','”','‘','’','—','-','-','<url>','@usuario'}
        self.model_order = model_order                        # n-gram model order.
        self.max_vocab_size = max_vocab_size                  # maximum vocabulary size.
        self.UNK, self.SOS, self.EOS = "<unk>", "<s>", "</s>" # special tokens.
        self.embeddings_model = embeddings_model              # embeddings model.
        self.remove_stopwords = remove_stopwords              # remove stopwords.
        self.vocab, self.w2id, self.id2w = None, None, None   # vocabulary, word to index, index to word mappings.
        self.embeddings_matrix = None                         # embeddings matrix.

    @staticmethod
    def default_tokenizer(doc: str) -> list:
        """
        Default tokenizer method. It splits the document by spaces.
        """
        return doc.split(" ")

    def remove_word(self, word: str) -> bool:
        """
        Remove word method. It removes punctuation and numbers. If remove_stopwords is True, it also removes stopwords.
        
        Parameters
        ----------
        word : str
            Word to remove.
        
        Returns
        -------
        bool
            True if the word is to remove, False otherwise.
        """
        word_lower = word.lower() # lowercase word.
        if self.remove_stopwords: # remove stopwords.
            return word_lower in self.punct or word_lower.isnumeric() or word_lower in stopwords_es
        else:
            return word_lower in self.punct or word_lower.isnumeric()

    def get_vocab(self, corpus: list) -> set:
        """
        Get vocabulary method in lowercase. It gets the vocabulary from the corpus based on the frequency distribution
        and the maximum vocabulary size. It also removes punctuation, numbers, and stopwords.

        Parameters
        ----------
        corpus : list
            List of documents.

        Returns
        -------
        set
            Set of words.
        """
        freq_dist = nltk.FreqDist(       # frequency distribution.
            w.lower() for doc in corpus  # lowercase document.
            for w in self.tokenizer(doc) # tokenize document.
            if not self.remove_word(w)   # remove word.
        )
        sorted_words = sorted(freq_dist, key = freq_dist.get, reverse = True)[:self.max_vocab_size - 3] # sorted words. -3 for special tokens.
        return set(sorted_words)

    def fit(self, corpus: list) -> None:
        """
        Fit method. Here we get the vocabulary and create the word to index and index to word mappings.
        If an embeddings model is provided, we also create the embeddings matrix.

        Parameters
        ----------
        corpus : list
            List of documents.
        """
        self.vocab = self.get_vocab(corpus) | {self.UNK, self.SOS, self.EOS} # vocabulary and add special tokens.
        self.w2id = {word: idx for idx, word in enumerate(self.vocab)}       # word to index.
        self.id2w = {idx: word for word, idx in self.w2id.items()}           # index to word.

        # If embeddings model is provided, create embeddings matrix.
        if self.embeddings_model is not None:
            self.embeddings_matrix = np.array(
                [self.embeddings_model[word] if word in self.embeddings_model else np.random.rand(self.embeddings_model.vector_size)
                for word in self.vocab]
                )

    def get_ngram_doc(self, doc: str) -> list:
        """
        Get n-gram document method. It gets the n-grams from the document.
        <s> padding before and </s> padding after each sentence.
        
        Parameters
        ----------
        doc : str
            Document.
            
        Returns
        -------
        list
            List of n-grams.
        """
        tokens = [token if token in self.vocab else self.UNK
                  for token in [w.lower() for w in self.tokenizer(doc)]] # lowercase and tokenization.
        tokens = (self.model_order - 1)*[self.SOS] + tokens + [self.EOS] # add start and end tokens.
        return list(nltk.ngrams(tokens, self.model_order))               # n-grams.

    def transform(self, corpus: list) -> Tuple[np.ndarray, np.ndarray]:
        """
        Transform method. Here we transform the corpus into n-grams.
        It builds the X (features) and y (labels) n-grams.
        
        Parameters
        ----------
        corpus : list
            List of documents.
    
        Returns
        -------
        Tuple[np.ndarray, np.ndarray]
            Tuple with X and y n-grams.
        """
        X_ngrams, y_ngrams = [], []                                     # n-grams.
        for doc in corpus:                                              # for each document.
            for words_window in self.get_ngram_doc(doc):                # for each n-gram.
                words_window_ids = [self.w2id[w] for w in words_window] # word to index.
                X_ngrams.append(words_window_ids[:-1])                  # X n-gram.
                y_ngrams.append(words_window_ids[-1])                   # y n-gram.    
        return np.array(X_ngrams), np.array(y_ngrams)                   # return X and y n-grams.


Create a instance of the above method.

In [4]:
tokenizer = TweetTokenizer(preserve_case = False, reduce_len = True, strip_handles = True) # tokenizer.
ngram_data = NgramData(model_order = 4,                # n-gram model order.
                       max_vocab_size = 5000,          # maximum vocabulary size.
                       tokenizer = tokenizer.tokenize) # tokenizer.
ngram_data.fit(X_train)                                # fit n-gram data object.
print("Vocabulary size:", len(ngram_data.vocab))       # vocabulary size.

Vocabulary size: 5000


Use the transform method:

In [5]:
X_ngrams_train, y_ngrams_train = ngram_data.transform(X_train)
X_ngrams_val, y_ngrams_val = ngram_data.transform(X_val)
print("Training observations: X =", X_ngrams_train.shape, "y =", y_ngrams_train.shape)
print("Validation observations: X =", X_ngrams_val.shape, "y =", y_ngrams_val.shape)

Training observations: X = (101339, 3) y = (101339,)
Validation observations: X = (11397, 3) y = (11397,)


Examples:

In [6]:
print("First 7 n-grams in training data:")
[[ngram_data.id2w[w] for w in tw] for tw in X_ngrams_train[:7]]

First 7 n-grams in training data:


[['<s>', '<s>', '<s>'],
 ['<s>', '<s>', 'q'],
 ['<s>', 'q', 'se'],
 ['q', 'se', 'puede'],
 ['se', 'puede', 'esperar'],
 ['puede', 'esperar', 'del'],
 ['esperar', 'del', 'maricon']]

In [7]:
print("First 7 following words in training data:")
[ngram_data.id2w[w] for w in y_ngrams_train[:7]]

First 7 following words in training data:


['q', 'se', 'puede', 'esperar', 'del', 'maricon', 'de']

# **2.-  Create the Data Loaders**

The code creates PyTorch datasets and data loaders to efficiently manage data batches:

In [8]:
# Arguments
batch_size = 64 # batch size.
num_workers = 2 # number of workers (defines parallel data-loading processes).

# Create data loaders.
train_dataset = TensorDataset(torch.tensor(X_ngrams_train), torch.tensor(y_ngrams_train)) # training data.
train_loader = DataLoader(dataset = train_dataset,   # training data.
                          batch_size = batch_size,   # batch size.
                          shuffle = True,            # if the data will be shuffled.
                          num_workers = num_workers) # number of workers.

# Validation
val_dataset = TensorDataset(torch.tensor(X_ngrams_val), torch.tensor(y_ngrams_val)) # validation data.
val_loader = DataLoader(dataset = val_dataset,     # validation data.
                        batch_size = batch_size,   # batch size.
                        shuffle = False,           # if the data will be shuffled.
                        num_workers = num_workers) # number of workers.

batch = next(iter(train_loader))
print("X shape:", batch[0].shape)
print("y shape:", batch[1].shape)

X shape: torch.Size([64, 3])
y shape: torch.Size([64])


In [9]:
print("First 7 n-grams in the first batch:")
[[ngram_data.id2w[w] for w in tw] for tw in batch[0].tolist()[:7]]

First 7 n-grams in the first batch:


[['putos', 'que', 'son'],
 ['caga', 'por', 'nada'],
 ['<unk>', 'con', 'el'],
 ['ni', 'le', 'pido'],
 ['que', 'partido', 'tan'],
 ['ver', 'la', 'vida'],
 ['qué', 'si', 'estaba']]

In [10]:
print("First 7 following words in the first batch:")
[ngram_data.id2w[w] for w in batch[1].tolist()[:7]]

First 7 following words in the first batch:


['algunos', '</s>', 'pelo', 'fotos', 'más', 'de', 'de']

# **3.- NeuralLanguageModel**

In [11]:
class NeuralLanguageModel(nn.Module):
    """
    Neural language model class.
    """
    def __init__(self, params):
        """
        Constructor method to initialize the neural language model.

        Parameters
        ----------
        params : dict
            Dictionary with parameters.

        Dictionary params
        -----------------
        vocab_size : int
            Vocabulary size.
        model_order : int
            Order of the n-gram model.
        embedding_size : int
            Vector size for the embeddings (each word is represented by a dense vector).
        hidden_size : int
            Number of neurons in the hidden layer.
        dropout : float
            Dropout.
        """
        super().__init__()
        self.window_size = params['model_order'] - 1   # window size.
        self.embedding_size = params['embedding_size'] # embedding size.

        # Embedding layer maps each word to a dense vector representation.
        self.embeddings = nn.Embedding(params['vocab_size'], self.embedding_size)  

        # Linear layers apply transformations to predict the next word.
        self.fc1 = nn.Linear(self.window_size * self.embedding_size, params['hidden_size'])

        # Dropout layer to prevent overfitting.
        self.drop1 = nn.Dropout(p = params['dropout'])  
    
        self.fc2 = nn.Linear(params['hidden_size'], params['vocab_size'], bias = False)     

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward method. Embeds input tokens into dense vectors. Reshapes them before feeding
        into the dense layers. Uses ReLU activation and Dropout before the final prediction.
        
        Parameters
        ----------
        x : torch.Tensor
            Input tensor.
            
        Returns
        -------
        torch.Tensor
            Output tensor. The model outputs raw scores (logits) for each word in the vocabulary.
        """
        x = self.embeddings(x).view(-1, self.window_size * self.embedding_size)
        return self.fc2(self.drop1(F.relu(self.fc1(x))))

# **4.- Utils**

In [12]:
class Utils:
    @staticmethod
    def get_preds(raw_logits: torch.Tensor) -> np.ndarray:
        """
        Get predictions method. Here we get the predictions from the raw logits (output of the neural network).
        Applies softmax to logits to obtain probabilities. Uses torch.argmax() to select the predicted class.

        Parameters
        ----------
        raw_logits : torch.Tensor
            Raw logits, i.e., the output of the neural network.

        Returns
        -------
        np.ndarray
            Predictions, i.e., the word with the highest probability.
        """
        probs = F.softmax(raw_logits.detach(), dim = 1)   # probabilities.
        return torch.argmax(probs, dim = 1).cpu().numpy() # predictions.


    @staticmethod
    def model_eval(data: DataLoader, model: nn.Module, gpu: bool) -> float:
        """
        Model evaluation method. Iterates over batches in the validation set.
        Accumulates predictions and computes accuracy using sklearn.

        Parameters
        ----------
        data : DataLoader
            Data loader.
        model : nn.Module
            Neural network model.
        gpu : bool
            If GPU is used.

        Returns
        -------
        float
            Accuracy score.
        """
        with torch.no_grad():                             # no gradients.
            preds, targets = [], []                       # predictions and targets.
            for window_words, labels in data:             # for each batch.
                if gpu:                                   # if GPU is used.
                    window_words = window_words.cuda()    # send to GPU.
                raw_logits = model(window_words)          # model output.
                preds.append(Utils.get_preds(raw_logits)) # predictions.
                targets.append(labels.numpy())            # targets.
        return accuracy_score([e for l in preds for e in l],[e for l in targets for e in l])
    
    @staticmethod
    def save_checkpoint(state: dict, is_best: bool, checkpoint_path: str, filename: str = "checkpoint.pth") -> None:
        """
        Save checkpoint method, i.e., saves the model's state during training. 

        Parameters
        ----------
        state : dict
            Dictionary with state, i.e., model state_dict, optimizer state_dict, epoch, loss, and accuracy.
        is_best : bool
            If is the best model, i.e., the model with the highest accuracy.
        checkpoint_path : str
            Checkpoint path.
        filename : str, optional
            Filename. Default is "checkpoint.pth".
        """
        filepath = os.path.join(checkpoint_path, filename)                             # file path.
        torch.save(state, filepath)                                                    # save checkpoint.                 
        if is_best:                                                                    # if is the best model.   
            shutil.copyfile(filepath, os.path.join(checkpoint_path, "model_best.pth")) # copy file.


# **5.- Trainer Class**

In [13]:
class Trainer:
    """
    Trainer class.
    """
    def __init__(self, model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, params: dict):
        """
        Constructor method for the Trainer class. It uses CrossEntropyLoss for multi-class classification and SGD optimizer.
        The learning rate scheduler is ReduceLROnPlateau (reduces learning rate when a metric has stopped improving).
        Uses patience-based early stopping: If no improvement occurs for several epochs, training halts.

        Parameters
        ----------
        model : nn.Module
            Neural network model.
        train_loader : DataLoader
            Training data loader.
        val_loader : DataLoader
            Validation data loader.
        params : dict
            Dictionary with parameters.

        Dictionary params
        -----------------
        lr : float
            Learning rate.
        num_epochs : int
            Number of epochs.
        patience : int
            Patience, i.e., early stopping threshold.
        lr_patience : int
            Learning rate patience.
        lr_factor : float
            Learning rate factor.
        savedir : str
            Save directory.
        use_gpu : bool
            Use GPU.
        """
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.params = params
        self.criterion = nn.CrossEntropyLoss()                                  # loss function.
        self.optimizer = torch.optim.SGD(model.parameters(), lr = params['lr']) # optimizer.
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(            # scheduler.
            self.optimizer,                   # optimizer to adjust.
            mode = 'min',                     # mode (min means reduce when the quantity monitored has stopped decreasing).
            patience = params['lr_patience'], # patience (number of epochs with no improvement after which learning rate will be reduced).
            factor = params['lr_factor']      # factor by which the learning rate will be reduced. new_lr = lr * factor.
        )

        # Metrics (in this case, accuracy).
        self.train_metric_history = [] # training metric history.
        self.val_metric_history = []   # validation metric history.
        self.best_val_metric = 0       # best validation accuracy.
    
    def train(self):
        """
        Train method. Here we train the neural network model.

        Description
        -----------
        The training process is as follows:
        1. We iterate over the number of epochs.
        2. For each epoch, we iterate over the training data.
        3. We calculate the loss and the accuracy.
        4. We update the weights.
        5. We evaluate the model on the validation data.
        6. We save the best model.
        7. We print the training and validation metrics.
        8. We break the training process if there is no improvement.
        9. We print the total time.
        """
        start_time = time.time() # start time.
        n_no_improve = 0         # number of epochs with no improvement.

        for epoch in tqdm(range(self.params['num_epochs']), desc = "Training..."):
            
            loss_epoch_batch = []   # Tracks the total loss for each batch in the current epoch.
            train_metric_batch = [] # Tracks the training accuracy for each batch in the current epoch.
            self.model.train()      # set model to training mode.
            
            for window_words, labels in self.train_loader:
                
                if self.params['use_gpu']:
                    window_words, labels = window_words.cuda(), labels.cuda()

                raw_logits = self.model(window_words)     # model output.
                loss = self.criterion(raw_logits, labels) # calculate loss.
                loss_epoch_batch.append(loss.item())      # append loss.

                preds = Utils.get_preds(raw_logits)                       # get predictions given the raw logits.
                targets = labels.cpu().numpy()                            # get targets.
                train_metric_batch.append(accuracy_score(targets, preds)) # append accuracy.

                self.optimizer.zero_grad() # zero gradients.
                loss.backward()            # backpropagation.
                self.optimizer.step()      # update weights.

            mean_train_metric = np.mean(train_metric_batch)     # mean training accuracy.
            self.train_metric_history.append(mean_train_metric) # append training accuracy.

            self.model.eval()                                                                  # set model to evaluation mode.
            val_metric = Utils.model_eval(self.val_loader, self.model, self.params['use_gpu']) # calculate validation accuracy.
            self.val_metric_history.append(val_metric)                                         # append validation accuracy.

            self.scheduler.step(val_metric) # adjust learning rate.

            # Check if there is an improvement.
            is_improvement = val_metric > self.best_val_metric
            if is_improvement:
                self.best_val_metric = val_metric
                n_no_improve = 0
            else:
                n_no_improve += 1

            # Save checkpoint.
            Utils.save_checkpoint(
                {
                    'epoch'           : epoch + 1,
                    'state_dict'      : self.model.state_dict(),
                    'best_val_metric' : self.best_val_metric,
                    'optimizer'       : self.optimizer.state_dict(),
                    'scheduler'       : self.scheduler.state_dict(),
                },
                is_improvement,
                self.params['savedir']
            )

            # Early stopping.
            if n_no_improve >= self.params['patience']:
                tqdm.write(f"No improvement. Breaking at epoch {epoch}")
                break

            tqdm.write(f"Epoch: {epoch + 1}/{self.params['num_epochs']} | " # print epoch.
                       f"Train acc: {mean_train_metric:.4f} | "             # print training accuracy.
                       f"Loss: {np.mean(loss_epoch_batch):.4f} | "          # print loss.
                       f"Val acc: {val_metric:.4f} | "                      # print validation accuracy.
                       f"Total Time: {time.time() - start_time:.2f} s")     # print total time.

        print(f"--- {time.time() - start_time:.2f} seconds ---")

## **5.1.- Training...**

In [14]:
# Parameters in only one dictionary.
params = {
    # NeuralLanguageModel parameters.
    "vocab_size"     : len(ngram_data.vocab),
    "model_order"    : 4,
    "embedding_size" : 100,
    "hidden_size"    : 200,
    "dropout"        : 0.1,
    # Trainer parameters.
    "lr"             : 2.3e-1,
    "num_epochs"     : 100,
    "patience"       : 20,
    "lr_patience"    : 10,
    "lr_factor"      : 0.5,
    "savedir"        : 'model',
    "use_gpu"        : torch.cuda.is_available()
}

# Create directory.
if not os.path.exists(params["savedir"]):           # if the directory does not exist.
    os.makedirs(params["savedir"], exist_ok = True) # create directory.

# Neural language model initialization.
model = NeuralLanguageModel(params = params)
if params["use_gpu"]:
    model.cuda()

# Training.
trainer = Trainer(model = model,               # neural network model.
                  train_loader = train_loader, # training data loader.
                  val_loader = val_loader,     # validation data loader.
                  params = params)             # parameters.
trainer.train()

Training...:   1%|          | 1/100 [00:08<13:37,  8.25s/it]

Epoch: 1/100 | Train acc: 0.1754 | Loss: 5.4872 | Val acc: 0.2129 | Total Time: 8.25 s


Training...:   2%|▏         | 2/100 [00:15<12:57,  7.94s/it]

Epoch: 2/100 | Train acc: 0.1850 | Loss: 5.0506 | Val acc: 0.1929 | Total Time: 15.97 s


Training...:   3%|▎         | 3/100 [00:23<12:45,  7.89s/it]

...

Epoch: 21/100 | Train acc: 0.2918 | Loss: 3.2011 | Val acc: 0.1987 | Total Time: 158.67 s


Training...:  22%|██▏       | 22/100 [02:45<09:22,  7.21s/it]

Epoch: 22/100 | Train acc: 0.2962 | Loss: 3.1663 | Val acc: 0.2204 | Total Time: 165.86 s


Training...:  22%|██▏       | 22/100 [02:53<10:13,  7.87s/it]

No improvement. Breaking at epoch 22
--- 173.06 seconds ---


# **Practice 7**

## **7.1.- Closest Words**

In [15]:
def print_k_closest_words(embeddings: nn.Embedding, word: str, top_k: int, ngram_data: NgramData) -> None:
    """
    Print closest words method. It prints the top k closest words to the word provided.

    Parameters
    ----------
    embeddings : nn.Embedding
        Pre-trained embeddings from the model.
    word : str
        Word.
    top_k : int
        Number of closest words.
    ngram_data : NgramData
        N-gram data.

    Description
    -----------
    This method prints the n closest words to the word provided.
    """
    word_id = torch.LongTensor([ngram_data.w2id[word]])                     # get word id.
    word_embeding = embeddings(word_id)                                     # Retrieves the embedding vector for the word from the model’s embedding layer.
    dists = torch.norm(embeddings.weight - word_embeding, dim = 1).detach() # calculate distances, embeddings.weight contains all word embeddings in the vocabulary.
    lst = sorted(enumerate(dists.numpy()), key = lambda x: x[1])            # sort by distance.
    for idx, difference in lst[1: top_k + 1]:                               # print the n closest words.
        print(ngram_data.id2w[idx], difference)

In [16]:
# Load best model.
best_model = NeuralLanguageModel(params)
best_model.load_state_dict(torch.load('model/model_best.pth', weights_only = True)['state_dict'])

# Print closest words.
word = "amlo"
print("-" * 50)
print("Closest words to:", word)
print("-" * 50)
print_k_closest_words(embeddings = best_model.embeddings,
                      word = word,
                      top_k = 10,
                      ngram_data = ngram_data)

--------------------------------------------------
Closest words to: amlo
--------------------------------------------------
psicopato 11.028707
🤧 11.100945
cartas 11.187885
papas 11.189248
v 11.204926
feminismo 11.307111
maneja 11.358217
equivoco 11.371642
horrible 11.374138
lozano 11.435538


## **7.2.- Generate sentence**

In [17]:
def parse_text(text: str, tokenizer: TweetTokenizer, ngram_data: NgramData) -> Tuple[list, list]:
    """
    Parse text method. It recieves a text and returns the tokens and token ids.

    Parameters
    ----------
    text : str
        Text to parse.
    tokenizer : TweetTokenizer
        Tokenizer.
    ngram_data : NgramData
        N-gram data.
    """
    all_tokens = [w.lower() if w in ngram_data.w2id else ngram_data.UNK for w in tokenizer(text)]     # Tokenizes the input text using tokenizer().
    token_ids = [ngram_data.w2id.get(w.lower(), ngram_data.w2id[ngram_data.UNK]) for w in all_tokens] # Maps tokens to their corresponding IDs.
    return all_tokens, token_ids                                                                      # Provides both the original tokens and their respective numeric IDs.

def sample_next_word(raw_logits: np.ndarray, temperature: float = 0.1) -> int:
    """
    Sample next word method. It receives the raw logits and returns the next word.
    It returns the index of the next word in the vocabulary.
    
    Parameters
    ----------
    raw_logits : np.ndarray
        Raw logits (output of the neural network).
    temperature : float, optional
        Temperature, i.e., the higher the temperature, the more random the output.
        The less the temperature, the more deterministic the output. Default is 0.1.
    
    Returns
    -------
    int
        Next word (index in the vocabulary).
    """
    raw_logits = np.asarray(raw_logits).astype('float64') # convert raw logits to numpy array.
    preds = raw_logits / temperature                      # apply temperature scaling.
    exp_preds = np.exp(preds - np.max(preds))             # Numeric stability fix
    preds = exp_preds / np.sum(exp_preds)                 # nomalize the probabilities.
    return np.random.choice(len(preds), p = preds)        # randomly samples one word ID from the distribution.

def predict_next_token(model: nn.Module, token_ids: list) -> int:
    """
    Predict next token method. It predicts the next token given the model and the token ids.

    Parameters
    ----------
    model : nn.Module
        Neural network model.
    token_ids : list
        Token ids list.
        
    Returns
    -------
    int
        Next token.
    """
    word_ids = torch.LongTensor(token_ids).unsqueeze(0)      # convert token IDs to tensor.
    raw_logits = model(word_ids).squeeze(0).detach().numpy() # raw predictions from the model.
    y_pred = sample_next_word(raw_logits, 1.0)               # to predict the next word.
    return y_pred

def generate_sentence(model: nn.Module, initial_text: str, tokenizer: TweetTokenizer, ngram_data: NgramData, max_length: int = 100) -> str:
    """
    This function generates a complete sentence using the trained model.
    It receives the initial text and predicts the next token word by word until the end of the sentence or a maximum length.
    
    Parameters
    ----------
    model : nn.Module
        Neural network model.
    initial_text : str
        Initial text to generate the sentence.
    tokenizer : TweetTokenizer
        Tokenizer.
    ngram_data : NgramData
        N-gram data.
    
    Returns
    -------
    str
        Generated sentence.
    """
    all_tokens, window_word_ids = parse_text(initial_text, tokenizer, ngram_data) # parse text.
    for _ in range(max_length):                                 # for each token.
        y_pred = predict_next_token(model, window_word_ids)     # predict next token.
        next_word = ngram_data.id2w.get(y_pred, ngram_data.UNK) # next word.
        all_tokens.append(next_word)                            # append next word.

        # If the next word is the end of sentence, break.
        if next_word == ngram_data.EOS: 
            break
        window_word_ids.pop(0)         # remove first word.
        window_word_ids.append(y_pred) # append next word.
    return " ".join(all_tokens)        # return generated sentence.

Examples:

In [18]:
initial_tokens = "<s> <s> <s>"
print("-"*50)
print("Learned embeddings:")
print("-"*50)
print(generate_sentence(best_model, initial_tokens, tokenizer.tokenize, ngram_data))

--------------------------------------------------
Learned embeddings:
--------------------------------------------------
<s> <s> <s> son tocó <unk> <unk> </s>


In [19]:
initial_tokens = "<s> <s> hola"
print("-"*50)
print("Learned embeddings:")
print("-"*50)
print(generate_sentence(best_model, initial_tokens, tokenizer.tokenize, ngram_data))

--------------------------------------------------
Learned embeddings:
--------------------------------------------------
<s> <s> hola de mi <unk> <unk> ya se ya no <unk> panocha hombres tonta <unk> <unk> <unk> <unk> <unk> hdp <unk> 😔 </s>


In [20]:
initial_tokens = "yo opino que"
print("-"*50)
print("Learned embeddings:")
print("-"*50)
print(generate_sentence(best_model, initial_tokens, tokenizer.tokenize, ngram_data))

--------------------------------------------------
Learned embeddings:
--------------------------------------------------
yo opino que una <unk> toda a escuchar <unk> </s>


## **7.3.- Log-likelihood**

In [25]:
def log_likelihood(model: nn.Module, text: str, ngram_model: NgramData) -> float:
    """
    Log likelihood method. This function calculates how likely a given text is according to the model.
    
    Parameters
    ----------
    model : nn.Module
        Neural network model.
    ngram_model : NgramData
        N-gram model.
    text : str
        Text to calculate the log likelihood.
    
    Returns
    -------
    float
        Log likelihood.
        The higher the log likelihood, the more likely the text is according to the model.
    """
    X, y = ngram_model.transform([text]) # transform text.
    if len(X) < 2:                       # if less than 2 tokens.
        return -np.inf                   # return negative infinity.

    X, y = X[2:], y[2:]
    X = torch.LongTensor(X).unsqueeze(0) # tensor.

    logits = model(X).detach()                 # logits.
    probs = F.softmax(logits, dim = 1).numpy() # probabilities.

    return np.sum([np.log(probs[i][w]) for i, w in enumerate(y)]) # log likelihood.

Examples

In [39]:
print(log_likelihood(model = best_model,
                     text = "Estamos en clase de lenguaje",
                     ngram_model = ngram_data))
print(log_likelihood(model = best_model,
                     text = "la natural Estamos clase caguama en de de lenguaje procesamiento de la",
                     ngram_model = ngram_data))

-17.562744
-50.899887


In [23]:
word_list = "sino gano me voy a la chingada".split(" ")
perms = [' '.join(p) for p in permutations(word_list)]
print("Permutations:", perms[:5])
print("Number of permutations:", len(perms))


Permutations: ['sino gano me voy a la chingada', 'sino gano me voy a chingada la', 'sino gano me voy la a chingada', 'sino gano me voy la chingada a', 'sino gano me voy chingada a la']
Number of permutations: 5040


In [24]:
# Calculate log likelihood for each permutation.
sortedstring = sorted([(log_likelihood(
    model = best_model,
    text = p,
    ngram_model = ngram_data
), p) for p in perms], reverse = True)

# Print top 5 and bottom 5.
print("-"*50)
for p, t in sortedstring[:5]:
    print(t, p)

print("-"*50)
for p, t in sortedstring[-5:]:
    print(t, p)

--------------------------------------------------
a la me chingada sino gano voy -6.2534313
a la voy me chingada sino gano -6.2584896
a la voy me sino chingada gano -6.3320637
me voy a sino chingada la gano -6.365067
me voy sino a la chingada gano -6.372312
--------------------------------------------------
chingada gano me voy la sino a -21.817842
chingada gano voy la me a sino -21.849598
chingada gano me voy a la sino -21.9718
chingada gano voy a sino la me -22.008034
chingada gano a me sino voy la -22.171154
